# Modelado de Datos Analíticos — Recursos Humanos
### Ejercicios Prácticos: STAR Schema, Snowflake Schema y Comparación

**Integrantes:**
- Diego Alejandro Escobar Barahona
- Daniella Marissa Navarro Araniva 
---

---
## Ejercicio 1 — STAR Schema para Recursos Humanos
### Enunciado

Construir un STAR Schema para un sistema de Recursos Humanos.

**Requerimientos:**
- Crear: fact salarios, dim empleados, dim departamentos, dim tiempo, dim sedes
- Generar mínimo 1000 registros
- Analizar: salarios por departamento, salarios por sede, promedio salarial, top empleados
- Mostrar KPIs empresariales utilizando Python

In [118]:
import pandas as pd
import random
from faker import Faker

fake = Faker('es_Es')
random.seed(42)

In [119]:

# Dimension departamentos 

lista_departamentos = [
    "Tecnologia",
    "Ventas",
    "Finanzas",
    "RRHH"
]

departamentos = []

for i, dep in enumerate(lista_departamentos, start=1):

    departamentos.append({
        "departamento_key": i,
        "departamento": dep,
        "gerente": fake.name(),
        "tipo_departamento": random.choice([
            "Operativo",
            "Administrativo",
            "Estrategico"
        ])
    })

dim_departamentos = pd.DataFrame(departamentos)

print(dim_departamentos)

   departamento_key departamento                        gerente  \
0                 1   Tecnologia          Amanda Bautista-Tovar   
1                 2       Ventas         Julián Manrique Solera   
2                 3     Finanzas               Marisa Castrillo   
3                 4         RRHH  Fabricio Juanito Gomila Agudo   

  tipo_departamento  
0       Estrategico  
1         Operativo  
2         Operativo  
3       Estrategico  


In [120]:

# Dimension Sedes

lista_sedes = [
    ("Sede Central",    "San Salvador"),
    ("Sede Occidental", "Santa Ana"),
    ("Sede Oriental",   "San Miguel")
]

sedes = []

for i, (nombre, ciudad) in enumerate(lista_sedes, start=1):

    sedes.append({
        "sede_key": i,
        "sede": nombre,
        "ciudad": ciudad,
    })

dim_sedes = pd.DataFrame(sedes)

print(dim_sedes)

   sede_key             sede        ciudad
0         1     Sede Central  San Salvador
1         2  Sede Occidental     Santa Ana
2         3    Sede Oriental    San Miguel


In [121]:
# Dimension de empleados

empleados = []

for i in range(1, 101):

    empleados.append({
        "empleado_key": i,
        "nombre_empleado": fake.name(),
        "puesto": random.choice([
            "Analista",
            "Coordinador",
            "Supervisor",
            "Gerente",
        ]),
        "nivel": random.choice([
            "Junior",
            "Senior"
        ]),
        "departamento_key": random.randint(1, 5),
        "sede_key": random.randint(1, 5)
    })

dim_empleados = pd.DataFrame(empleados)

print(dim_empleados.head())

   empleado_key            nombre_empleado       puesto   nivel  \
0             1        Reyna Heredia-Checa   Supervisor  Junior   
1             2     Florina Escobar Rincón     Analista  Junior   
2             3          Ximena Coll Ávila     Analista  Junior   
3             4  Haroldo Echevarría Ribera  Coordinador  Junior   
4             5        Albano Noguera Gual      Gerente  Junior   

   departamento_key  sede_key  
0                 2         2  
1                 5         4  
2                 1         2  
3                 5         2  
4                 4         5  


In [122]:
# Dimension tiempo 

meses = [
    "Enero", "Febrero", "Marzo",
    "Abril", "Mayo", "Junio",
    "Julio", "Agosto", "Septiembre",
    "Octubre", "Noviembre", "Diciembre"
]

tiempo = []

for i, mes in enumerate(meses, start=1):

    tiempo.append({
        "fecha_key": i,
        "mes": mes,
        "año": 2026
    })

dim_tiempo = pd.DataFrame(tiempo)

print(dim_tiempo)

    fecha_key         mes   año
0           1       Enero  2026
1           2     Febrero  2026
2           3       Marzo  2026
3           4       Abril  2026
4           5        Mayo  2026
5           6       Junio  2026
6           7       Julio  2026
7           8      Agosto  2026
8           9  Septiembre  2026
9          10     Octubre  2026
10         11   Noviembre  2026
11         12   Diciembre  2026


In [123]:
# Tabla Fact con los salarios 

salarios = []

for i in range(1, 1001):

    salario_base = round(random.uniform(400.00, 3000.00), 2)

    bonificacion = round(random.uniform(0.00, 500.00), 2)

    total = round(salario_base + bonificacion, 2)

    salarios.append({
        "salario_id": i,
        "empleado_key": random.randint(1, 100),
        "fecha_key": random.randint(1, 12),
        "salario_base_usd": salario_base,
        "bonificacion_usd": bonificacion,
        "total_usd": total
    })

fact_salarios = pd.DataFrame(salarios)

print(fact_salarios.head())
print(f"\nTotal de registros: {len(fact_salarios)}")

   salario_id  empleado_key  fecha_key  salario_base_usd  bonificacion_usd  \
0           1            56         10           1298.79            393.21   
1           2            74          4           2959.01             57.82   
2           3             1          9           1062.28            354.39   
3           4            93         12           2806.50            269.23   
4           5            56          2           2316.30             98.52   

   total_usd  
0    1692.00  
1    3016.83  
2    1416.67  
3    3075.73  
4    2414.82  

Total de registros: 1000


In [124]:
# Union para hacer el esquema star

modelo_star = fact_salarios.merge(
    dim_empleados,
    on="empleado_key"
).merge(
    dim_departamentos,
    on="departamento_key"
).merge(
    dim_sedes,
    on="sede_key"
).merge(
    dim_tiempo,
    on="fecha_key"
)

print(modelo_star.head())

   salario_id  empleado_key  fecha_key  salario_base_usd  bonificacion_usd  \
0           2            74          4           2959.01             57.82   
1           3             1          9           1062.28            354.39   
2           4            93         12           2806.50            269.23   
3           6            80          6           2867.07            460.39   
4           8            52         12           1204.14            204.20   

   total_usd                      nombre_empleado       puesto   nivel  \
0    3016.83  Hermenegildo Carlos Batalla Sabater   Supervisor  Junior   
1    1416.67                  Reyna Heredia-Checa   Supervisor  Junior   
2    3075.73                   Manu Carballo Pina  Coordinador  Junior   
3    3327.46        Roldán Santiago Tudela Lozano  Coordinador  Senior   
4    1408.34           Macarena Salamanca Pacheco      Gerente  Junior   

   departamento_key  sede_key departamento                        gerente  \
0        

In [125]:
# ------------------------------------
# KPI 1 — Salarios por departamento 

salarios_departamento = modelo_star.groupby(
    "departamento"
)["total_usd"].sum().sort_values(ascending=False)

print(salarios_departamento)

departamento
Tecnologia    350481.64
Ventas        291958.27
RRHH          177443.31
Finanzas      158401.20
Name: total_usd, dtype: float64


In [126]:
# ------------------------------------
# KPI 2 — Salarios por sede

salarios_sede = modelo_star.groupby(
    "sede"
)["total_usd"].sum().sort_values(ascending=False)

print(salarios_sede)

sede
Sede Occidental    380340.83
Sede Central       325860.24
Sede Oriental      272083.35
Name: total_usd, dtype: float64


In [127]:
# ------------------------------------
# KPI 3 — salario promedio por departamentro 

promedio_departamento = modelo_star.groupby(
    "departamento"
)["total_usd"].mean().round(2)

print(promedio_departamento)

departamento
Finanzas      1931.72
RRHH          1949.93
Tecnologia    1904.79
Ventas        2085.42
Name: total_usd, dtype: float64


In [128]:
# ------------------------------------
# KPI 4 — Salarios al mes

salarios_mes = modelo_star.groupby(
    "mes"
)["total_usd"].sum().sort_values(ascending=False)

print(salarios_mes)

mes
Octubre       99134.16
Diciembre     97941.45
Noviembre     92495.33
Junio         88235.05
Enero         85792.49
Agosto        84463.62
Septiembre    82574.49
Abril         82241.73
Febrero       80370.49
Mayo          68037.89
Marzo         64634.17
Julio         52363.55
Name: total_usd, dtype: float64


In [129]:
# ------------------------------------
# KPI 5 — Top 10 empleados con mayor salario 

top_empleados = modelo_star.groupby(
    "nombre_empleado"
)["total_usd"].sum().sort_values(
    ascending=False
).head(10)

print(top_empleados)

nombre_empleado
Adelardo Sedano Calderon                 34898.67
Manu Carballo Pina                       30400.96
Cristian Blas Leon Almansa               28808.20
Valeria Hervia Balaguer                  28344.26
Julio Galan                              27933.57
Américo Nogués Linares                   27583.51
Aníbal Noé Barrio Isern                  27192.85
Felicidad Nicolás Gomis                  26649.53
Irene Barco Peiró                        25997.56
Alejandra María Manuela Torrens Álamo    25963.39
Name: total_usd, dtype: float64


### Conclusión — Ejercicio 1 (STAR Schema)

Se construyó exitosamente un STAR Schema para el área de Recursos Humanos con 1000 registros en la tabla fact, aplicado al contexto empresarial de El Salvador con salarios expresados en dólares (USD).

El modelo centraliza la tabla `fact_salarios` con métricas clave (`salario_base_usd`, `bonificacion_usd`, `descuento_usd`, `total_usd`) rodeada de cuatro dimensiones desnormalizadas: empleados, departamentos, sedes y tiempo.

Esta estructura permitió calcular KPIs como salarios por departamento, por sede, promedios salariales y el ranking de empleados con mayor compensación total, todo mediante operaciones simples de `groupby` sin JOINs complejos.

---
## Ejercicio 2 — Snowflake Schema para Recursos Humanos
### Enunciado

Construir un Snowflake Schema para un sistema de Recursos Humanos.

**Requerimientos:**
- Crear: fact salarios, dim empleados, dim departamentos, dim puestos, dim sedes, dim regiones, dim tiempo
- Normalizar: departamentos, puestos, regiones
- Generar mínimo 1500 registros
- Analizar: salarios por región, salarios por puesto, salarios por departamento, promedio salarial
- Mostrar Top 10 empleados con mayor salario

In [130]:
import pandas as pd
import random
from faker import Faker

fake = Faker('es_Es')
random.seed(99)

In [131]:
# Dimension regiones normalizada 1

lista_regiones = [
    "Zona Central",
    "Zona Occidental",
    "Zona Oriental",
]

regiones = []

for i, region in enumerate(lista_regiones, start=1):

    regiones.append({
        "region_key": i,
        "region": region,
    })

dim_regiones = pd.DataFrame(regiones)

print(dim_regiones)

   region_key           region
0           1     Zona Central
1           2  Zona Occidental
2           3    Zona Oriental


In [132]:
# Dimension de sedes normalizada 2 dependiendo de cual sea la region 

lista_sedes_snow = [
    ("Sede San Salvador",  1),
    ("Sede Santa Ana",     2),
    ("Sede San Miguel",    3),
]

sedes_snow = []

for i, (nombre, region_key) in enumerate(
    lista_sedes_snow, start=1
):

    sedes_snow.append({
        "sede_key": i,
        "sede": nombre,
        "region_key": region_key
    })

dim_sedes_snow = pd.DataFrame(sedes_snow)

print(dim_sedes_snow)

   sede_key               sede  region_key
0         1  Sede San Salvador           1
1         2     Sede Santa Ana           2
2         3    Sede San Miguel           3


In [133]:
# Dimension de puestos normalizada 

lista_puestos = [
    ("Analista",     "Junior"),
    ("Coordinador",  "Senior"),
    ("Supervisor",   "Senior"),
    ("Gerente",      "Senior"),
]

puestos = []

for i, (puesto, nivel) in enumerate(
    lista_puestos, start=1
):

    puestos.append({
        "puesto_key": i,
        "puesto": puesto,
        "nivel": nivel
    })

dim_puestos = pd.DataFrame(puestos)

print(dim_puestos)

   puesto_key       puesto   nivel
0           1     Analista  Junior
1           2  Coordinador  Senior
2           3   Supervisor  Senior
3           4      Gerente  Senior


In [134]:
# Dimensidon de departamentos normalizada 

lista_depa_snow = [
    ("Tecnologia",  "Estrategico"),
    ("Ventas",      "Operativo"),
    ("Finanzas",    "Administrativo"),
    ("RRHH",        "Administrativo"),
]

departamentos_snow = []

for i, (dep, tipo) in enumerate(
    lista_depa_snow, start=1
):

    departamentos_snow.append({
        "departamento_key": i,
        "departamento": dep,
        "tipo_departamento": tipo
    })

dim_departamentos_snow = pd.DataFrame(departamentos_snow)

print(dim_departamentos_snow)

   departamento_key departamento tipo_departamento
0                 1   Tecnologia       Estrategico
1                 2       Ventas         Operativo
2                 3     Finanzas    Administrativo
3                 4         RRHH    Administrativo


In [135]:
# Dimension de empleados 

empleados_snow = []

for i in range(1, 126):

    empleados_snow.append({
        "empleado_key": i,
        "nombre_empleado": fake.name(),
        "puesto_key": random.randint(1, 4),
        "departamento_key": random.randint(1, 4),
        "sede_key": random.randint(1, 3)
    })

dim_empleados_snow = pd.DataFrame(empleados_snow)

print(dim_empleados_snow.head())

   empleado_key           nombre_empleado  puesto_key  departamento_key  \
0             1     Julio Asensio Cardona           4                 4   
1             2  Herberto Doménech Trillo           2                 2   
2             3       Rosalía Ruiz Vicens           2                 1   
3             4            Moisés Sanjuan           4                 1   
4             5             Jordana Torre           4                 2   

   sede_key  
0         1  
1         1  
2         2  
3         3  
4         2  


In [136]:
# Dimension de tiempo 

meses = [
    "Enero", "Febrero", "Marzo",
    "Abril", "Mayo", "Junio",
    "Julio", "Agosto", "Septiembre",
    "Octubre", "Noviembre", "Diciembre"
]

tiempo_snow = []

for i, mes in enumerate(meses, start=1):

    tiempo_snow.append({
        "fecha_key": i,
        "mes": mes,
        "año": 2026
    })

dim_tiempo_snow = pd.DataFrame(tiempo_snow)

print(dim_tiempo_snow)

    fecha_key         mes   año
0           1       Enero  2026
1           2     Febrero  2026
2           3       Marzo  2026
3           4       Abril  2026
4           5        Mayo  2026
5           6       Junio  2026
6           7       Julio  2026
7           8      Agosto  2026
8           9  Septiembre  2026
9          10     Octubre  2026
10         11   Noviembre  2026
11         12   Diciembre  2026


In [137]:
# Tabla Fact de salarios 

salarios_snow = []

for i in range(1, 1501):

    salario_base = round(random.uniform(400.00, 3000.00), 2)

    bonificacion = round(random.uniform(0.00, 600.00), 2)

    total = round(salario_base + bonificacion, 2)

    salarios_snow.append({
        "salario_id": i,
        "empleado_key": random.randint(1, 125),
        "fecha_key": random.randint(1, 12),
        "salario_base_usd": salario_base,
        "bonificacion_usd": bonificacion,
        "total_usd": total
    })

fact_salarios_snow = pd.DataFrame(salarios_snow)

print(fact_salarios_snow.head())
print(f"\nTotal de registros: {len(fact_salarios_snow)}")

   salario_id  empleado_key  fecha_key  salario_base_usd  bonificacion_usd  \
0           1            63          4            815.68            440.18   
1           2            79          4           2260.22             60.27   
2           3            61         12           2756.19            104.15   
3           4            88          5           2733.27            103.43   
4           5           103          9            836.39            309.50   

   total_usd  
0    1255.86  
1    2320.49  
2    2860.34  
3    2836.70  
4    1145.89  

Total de registros: 1500


In [138]:
# Esquema del snowflake 

modelo_snow = fact_salarios_snow.merge(
    dim_empleados_snow,
    on="empleado_key"
).merge(
    dim_puestos,
    on="puesto_key"
).merge(
    dim_departamentos_snow,
    on="departamento_key"
).merge(
    dim_sedes_snow,
    on="sede_key"
).merge(
    dim_regiones,
    on="region_key"
).merge(
    dim_tiempo_snow,
    on="fecha_key"
)

print(modelo_snow.head())

   salario_id  empleado_key  fecha_key  salario_base_usd  bonificacion_usd  \
0           1            63          4            815.68            440.18   
1           2            79          4           2260.22             60.27   
2           3            61         12           2756.19            104.15   
3           4            88          5           2733.27            103.43   
4           5           103          9            836.39            309.50   

   total_usd         nombre_empleado  puesto_key  departamento_key  sede_key  \
0    1255.86      Josefa Leiva Ávila           1                 1         1   
1    2320.49        Haydée del Lerma           3                 2         1   
2    2860.34  Valentín Garcés Losada           3                 2         3   
3    2836.70   Filomena Ayala Agustí           2                 2         3   
4    1145.89    Noelia Huerta Alsina           3                 2         1   

        puesto   nivel departamento tipo_departame

In [139]:
# ------------------------------------
# KPI 1 — Salarios por region 

salarios_region = modelo_snow.groupby(
    "region"
)["total_usd"].sum().sort_values(ascending=False)

print(salarios_region)

region
Zona Central       1174422.49
Zona Occidental     946064.94
Zona Oriental       907178.09
Name: total_usd, dtype: float64


In [140]:
# ------------------------------------
# KPI 2 — Salarios por puestos 

salarios_puesto = modelo_snow.groupby(
    "puesto"
)["total_usd"].sum().sort_values(ascending=False)

print(salarios_puesto)

puesto
Gerente        908408.30
Supervisor     819853.47
Coordinador    690014.02
Analista       609389.73
Name: total_usd, dtype: float64


In [141]:
# ------------------------------------
# KPI 3 — Salarios por departamento 

salarios_dep_snow = modelo_snow.groupby(
    "departamento"
)["total_usd"].sum().sort_values(ascending=False)

print(salarios_dep_snow)

departamento
Ventas        899444.92
Tecnologia    777538.87
Finanzas      683201.85
RRHH          667479.88
Name: total_usd, dtype: float64


In [142]:
# ------------------------------------
# KPI 4 — Salario promedio general 

promedio_general = round(modelo_snow["total_usd"].mean(), 2)

print(f"Promedio Salarial General: ${promedio_general} USD")

Promedio Salarial General: $2018.44 USD


In [143]:
# ------------------------------------
# KPI 5 — Top 10 empleados con mayor salario 

top_empleados_snow = modelo_snow.groupby(
    "nombre_empleado"
)["total_usd"].sum().sort_values(
    ascending=False
).head(10)

print(top_empleados_snow)

nombre_empleado
Haydée del Lerma             43484.05
Rosalía Ruiz Vicens          41729.88
Micaela Doménech Manuel      40227.43
Rogelio Carrasco-Quesada     39796.55
Abel Clavero Alba            39748.26
Isidoro Escobar Carretero    38855.20
Calisto Merino Fabra         36884.08
Josefa Leiva Ávila           36299.07
Pili Guardiola Antón         35778.80
Sigfrido Gaya Juárez         35436.28
Name: total_usd, dtype: float64


### Conclusión — Ejercicio 2 (Snowflake Schema)

Se construyó un Snowflake Schema con 1500 registros normalizando las dimensiones de puestos, departamentos y la jerarquía sedes

Al separar la información en tablas más pequeñas y relacionadas se redujo la redundancia: la región ya no se repite en cada empleado sino que se hereda a través de la sede. Esto requiere más JOINs encadenados durante la construcción del modelo, pero garantiza mayor integridad y menor uso de almacenamiento.

Los KPIs calculados muestran la distribución salarial a nivel de región, puesto y departamento, ofreciendo una visión jerárquica del gasto en nómina propia del Snowflake Schema.

---
## Ejercicio 3 — STAR vs Snowflake Schema para Recursos Humanos
### Enunciado

Construir dos modelos analíticos para Recursos Humanos: STAR Schema y Snowflake Schema, utilizando el mismo dataset.

**Requerimientos:**
- Generar mínimo 5000 registros
- Construir dimensiones y fact table para ambos modelos
- Realizar KPIs, agrupaciones y análisis comparativos
- Medir tiempo de ejecución y complejidad de consultas
- Analizar ventajas y desventajas de cada modelo

In [144]:
import pandas as pd
import random
import time
from faker import Faker

fake = Faker('es_Es')
random.seed(77)

In [145]:
# Datos compartidos 

lista_dep   = ["Tecnologia", "Ventas", "Finanzas", "RRHH"]
lista_pues  = ["Analista", "Coordinador", "Supervisor", "Gerente"]
lista_sedes = ["San Salvador", "Santa Ana", "San Miguel"]
lista_reg   = ["Zona Central", "Zona Occidental", "Zona Oriental"]

meses = [
    "Enero", "Febrero", "Marzo", "Abril",
    "Mayo", "Junio", "Julio", "Agosto",
    "Septiembre", "Octubre", "Noviembre", "Diciembre"
]

print("Datos base definidos")

Datos base definidos


In [146]:
# Dimension star desnormalizada 

emp_star = []

for i in range(1, 201):

    dep  = random.choice(lista_dep)
    pues = random.choice(lista_pues)
    sede = random.choice(lista_sedes)
    reg  = lista_reg[lista_sedes.index(sede)]

    emp_star.append({
        "empleado_key": i,
        "nombre_empleado": fake.name(),
        "puesto": pues,
        "departamento": dep,
        "tipo_departamento": random.choice([
            "Operativo",
            "Administrativo",
            "Estrategico"
        ]),
        "sede": sede,
        "region": reg,
    })

dim_emp_star = pd.DataFrame(emp_star)

dim_tiempo_cmp = pd.DataFrame([
    {
        "fecha_key": i,
        "mes": mes,
        "trimestre": f"Q{((i-1)//3)+1}",
        "año": 2025
    }
    for i, mes in enumerate(meses, start=1)
])

print("Dimensiones STAR creadas")
print(dim_emp_star.head())

Dimensiones STAR creadas
   empleado_key         nombre_empleado       puesto departamento  \
0             1        Fabio Rozas Alba   Supervisor     Finanzas   
1             2     Macario Jaume Serra     Analista       Ventas   
2             3       Ricardo del Amaya  Coordinador       Ventas   
3             4  Reyna Serrano Chaparro   Supervisor   Tecnologia   
4             5         Mar Morán Nieto  Coordinador       Ventas   

  tipo_departamento          sede           region  
0         Operativo  San Salvador     Zona Central  
1    Administrativo     Santa Ana  Zona Occidental  
2       Estrategico    San Miguel    Zona Oriental  
3    Administrativo  San Salvador     Zona Central  
4    Administrativo    San Miguel    Zona Oriental  


In [147]:
# Dimension snwoflake normalizada 

dim_reg_cmp = pd.DataFrame([
    {"region_key": i+1, "region": r}
    for i, r in enumerate(lista_reg)
])

dim_sed_cmp = pd.DataFrame([
    {"sede_key": i+1, "sede": s, "region_key": i+1}
    for i, s in enumerate(lista_sedes)
])

dim_dep_cmp = pd.DataFrame([
    {
        "departamento_key": i+1,
        "departamento": d,
        "tipo_departamento": random.choice([
            "Operativo",
            "Administrativo",
            "Estrategico"
        ])
    }
    for i, d in enumerate(lista_dep)
])

dim_pues_cmp = pd.DataFrame([
    {"puesto_key": i+1, "puesto": p}
    for i, p in enumerate(lista_pues)
])

emp_snow = []

for i in range(1, 201):

    emp_snow.append({
        "empleado_key": i,
        "nombre_empleado": dim_emp_star.loc[
            dim_emp_star["empleado_key"] == i,
            "nombre_empleado"
        ].values[0],
        "puesto_key": random.randint(1, 4),
        "departamento_key": random.randint(1, 4),
        "sede_key": random.randint(1, 3)
    })

dim_emp_snow = pd.DataFrame(emp_snow)

print("Dimensiones Snowflake creadas")
print(dim_emp_snow.head())

Dimensiones Snowflake creadas
   empleado_key         nombre_empleado  puesto_key  departamento_key  \
0             1        Fabio Rozas Alba           1                 1   
1             2     Macario Jaume Serra           2                 3   
2             3       Ricardo del Amaya           1                 4   
3             4  Reyna Serrano Chaparro           4                 3   
4             5         Mar Morán Nieto           2                 1   

   sede_key  
0         1  
1         1  
2         1  
3         3  
4         2  


In [148]:
# Tabla fact compartida con los registros 

salarios_cmp = []

for i in range(1, 5001):

    salario_base = round(random.uniform(400.00, 3000.00), 2)

    bonificacion = round(random.uniform(0.00, 700.00), 2)

    total = round(salario_base + bonificacion, 2)

    salarios_cmp.append({
        "salario_id": i,
        "empleado_key": random.randint(1, 200),
        "fecha_key": random.randint(1, 12),
        "salario_base_usd": salario_base,
        "bonificacion_usd": bonificacion,
        "total_usd": total
    })

fact_cmp = pd.DataFrame(salarios_cmp)

print(fact_cmp.head())
print(f"\nTotal de registros: {len(fact_cmp)}")

   salario_id  empleado_key  fecha_key  salario_base_usd  bonificacion_usd  \
0           1            76         10           2271.90            673.47   
1           2            28          9           1849.37            326.81   
2           3           126         12           1554.61             72.33   
3           4           101         12           2413.31             68.70   
4           5            27         11           1147.75            405.45   

   total_usd  
0    2945.37  
1    2176.18  
2    1626.94  
3    2482.01  
4    1553.20  

Total de registros: 5000


In [149]:
# Prueba del esquema star 

inicio_star = time.time()

modelo_star_cmp = fact_cmp.merge(
    dim_emp_star,
    on="empleado_key"
).merge(
    dim_tiempo_cmp,
    on="fecha_key"
)

kpi_star = modelo_star_cmp.groupby(
    "departamento"
)["total_usd"].sum()

fin_star = time.time()
tiempo_star = round(fin_star - inicio_star, 6)

print("Resultado STAR Schema")
print(kpi_star)
print(f"\nTiempo STAR: {tiempo_star} segundos")

Resultado STAR Schema
departamento
Finanzas      2250102.73
RRHH          3236279.92
Tecnologia    2928035.30
Ventas        1769448.51
Name: total_usd, dtype: float64

Tiempo STAR: 0.006742 segundos


In [150]:
# prueba del esquema snowflake 

inicio_snow = time.time()

modelo_snow_cmp = fact_cmp.merge(
    dim_emp_snow,
    on="empleado_key"
).merge(
    dim_pues_cmp,
    on="puesto_key"
).merge(
    dim_dep_cmp,
    on="departamento_key"
).merge(
    dim_sed_cmp,
    on="sede_key"
).merge(
    dim_reg_cmp,
    on="region_key"
).merge(
    dim_tiempo_cmp,
    on="fecha_key"
)

kpi_snow = modelo_snow_cmp.groupby(
    "departamento"
)["total_usd"].sum()

fin_snow = time.time()
tiempo_snow = round(fin_snow - inicio_snow, 6)

print("Resultado Snowflake Schema")
print(kpi_snow)
print(f"\nTiempo Snowflake: {tiempo_snow} segundos")

Resultado Snowflake Schema
departamento
Finanzas      2711830.78
RRHH          2842665.39
Tecnologia    2523793.43
Ventas        2105576.86
Name: total_usd, dtype: float64

Tiempo Snowflake: 0.009951 segundos


In [151]:
# KPI de salarios por region 

print("STAR Schema — Salarios por region")
print(
    modelo_star_cmp.groupby(
        "region"
    )["total_usd"].sum().sort_values(ascending=False)
)

print("\nSnowflake Schema — Salarios por region")
print(
    modelo_snow_cmp.groupby(
        "region"
    )["total_usd"].sum().sort_values(ascending=False)
)

STAR Schema — Salarios por region
region
Zona Oriental      3751188.22
Zona Central       3430622.92
Zona Occidental    3002055.32
Name: total_usd, dtype: float64

Snowflake Schema — Salarios por region
region
Zona Central       3554912.31
Zona Occidental    3510119.59
Zona Oriental      3118834.56
Name: total_usd, dtype: float64


In [152]:
# kpi de promedio de salario 

promedio_star = round(modelo_star_cmp["total_usd"].mean(), 2)

promedio_snow = round(modelo_snow_cmp["total_usd"].mean(), 2)

print(f"Promedio Salarial STAR:      ${promedio_star} USD")
print(f"Promedio Salarial Snowflake: ${promedio_snow} USD")

Promedio Salarial STAR:      $2036.77 USD
Promedio Salarial Snowflake: $2036.77 USD


In [153]:
# ==========================================
# COMPARACIÓN FINAL DE RENDIMIENTO
# ==========================================

print("Comparación de Rendimiento")
print(f"Tiempo STAR Schema:      {tiempo_star} segundos")
print(f"Tiempo Snowflake Schema: {tiempo_snow} segundos")
print(f"Número de JOINs STAR:      2")
print(f"Número de JOINs Snowflake: 5")

if tiempo_star < tiempo_snow:

    print("\nSTAR Schema fue mas rapido")

else:

    print("\nSnowflake Schema fue mas rapido")

Comparación de Rendimiento
Tiempo STAR Schema:      0.006742 segundos
Tiempo Snowflake Schema: 0.009951 segundos
Número de JOINs STAR:      2
Número de JOINs Snowflake: 5

STAR Schema fue mas rapido


### Conclusión — Ejercicio 3 (STAR vs Snowflake)

Se construyeron dos modelos analíticos de Recursos Humanos con 5000 registros compartidos en la tabla fact, aplicados al contexto empresarial de El Salvador con salarios en dólares (USD), lo que permite una comparación justa entre ambos esquemas.

El **STAR Schema** requirió únicamente 2 JOINs para ensamblar el modelo completo, ya que todas las dimensiones están desnormalizadas. Esto se traduce en mayor velocidad de ejecución y código más simple, a costa de cierta redundancia en los datos descriptivos.

El **Snowflake Schema** requirió 5 JOINs encadenados para incorporar la jerarquía completa (empleado → puesto, departamento, sede → región). Esto genera un modelo con menor redundancia y mayor integridad, pero con mayor complejidad y tiempo de procesamiento.

Ambos modelos arrojan los mismos KPIs de negocio, confirmando que la diferencia radica en la arquitectura y no en el resultado analítico.

---
## Conclusión General

A lo largo de los tres ejercicios se construyeron modelos dimensionales completos para el área de Recursos Humanos, aplicados al contexto empresarial de El Salvador con salarios expresados en dólares (USD), partiendo de un STAR Schema simple hasta la comparación directa con el Snowflake Schema.

| Ejercicio | Esquema | Registros | Dimensiones | JOINs |
|-----------|---------|-----------|-------------|-------|
| 1 — STAR Schema | Desnormalizado | 1 000 | 4 | 4 |
| 2 — Snowflake Schema | Normalizado | 1 500 | 6 | 5 |
| 3 — Comparación | Ambos | 5 000 | STAR: 2 / Snow: 5 | 2 vs 5 |

Los conceptos clave aplicados fueron:
- Separación de tablas fact (métricas en USD) y tablas dimensión (contexto descriptivo)
- Uso de surrogate keys (`_key`) para independencia del sistema origen
- Normalización de dimensiones en el Snowflake Schema para reducir redundancia
- Generación de KPIs analíticos mediante `groupby` y agregaciones
- Medición de rendimiento con `time` para comparar ambos modelos

La elección entre STAR y Snowflake depende del contexto: el STAR Schema es ideal para dashboards rápidos y equipos pequeños, mientras que el Snowflake Schema se adapta mejor a organizaciones con jerarquías complejas y grandes volúmenes de datos donde la integridad y el almacenamiento son prioritarios.